# Week 8：PCA 接入分类 Pipeline

目标：在相同的交叉验证规则下，对比逻辑回归直接使用原始特征，与“标准化 → PCA → 逻辑回归”的流程。

## 为什么必须把 PCA 放在 Pipeline 里

PCA 会从数据中学习主方向；标准化也会学习均值和标准差。做交叉验证时，它们只能在每一折的训练部分学习，随后再转换该折验证部分。

Pipeline 会自动保证这个顺序，避免把验证折的信息提前泄露给 PCA。

In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

# 仅在开发数据上比较流程，封存测试集不参与这次选择。
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [3]:
baseline = Pipeline([
    # 逻辑回归需要尺度一致的输入。
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

pca_pipeline = Pipeline([
    # 先标准化，再让 PCA 学习主方向。
    ('scaler', StandardScaler()),
    # 自动选择能累计解释至少 95% 方差的最少主成分。
    ('pca', PCA(n_components=0.95, random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

In [4]:
rows = []
for name, model in {
    'standardized logistic regression': baseline,
    'standardized PCA(95%) + logistic regression': pca_pipeline,
}.items():
    scores = cross_validate(
        model, X_train, y_train, cv=cv, scoring='accuracy',
        return_train_score=True, n_jobs=-1,
    )
    rows.append({
        'pipeline': name,
        'train_mean': scores['train_score'].mean(),
        'validation_mean': scores['test_score'].mean(),
        'validation_std': scores['test_score'].std(),
    })

comparison = pd.DataFrame(rows)
comparison['gap'] = comparison['train_mean'] - comparison['validation_mean']
display(comparison)

,pipeline,train_mean,validation_mean,validation_std,gap
0,standardized logistic regression,0.989011,0.978022,0.009829,0.010989
1,standardized PCA(95%) + logistic regression,0.988462,0.982418,0.005383,0.006044


In [5]:
# 为了便于解释，本单元在完整开发集上单独拟合 PCA，只查看最终保留了多少维。
pca_pipeline.fit(X_train, y_train)
pca_step = pca_pipeline.named_steps['pca']
print(f'完整开发集上，PCA 将 {X.shape[1]} 个特征压缩为 {pca_step.n_components_} 个主成分')
print(f'累计解释方差：{pca_step.explained_variance_ratio_.sum():.3f}')

完整开发集上，PCA 将 30 个特征压缩为 10 个主成分
累计解释方差：0.953


## 如何做决策

- 若 PCA 后验证效果更好或相近，但特征维度显著下降：PCA 可能值得使用。
- 若 PCA 后验证效果明显下降：不要为了“降维”而使用 PCA。
- 即使效果相近，也要考虑解释性：主成分通常比原始医学特征更难解释。

思考题：为什么 PCA 放在 `StandardScaler` 后面，而不是前面？